<a href="https://colab.research.google.com/github/Crayon897/miami-dade-311-analysis/blob/main/311_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Which Miami-Dade ZIP Codes Wait Longest for County Services?

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/311_2023.csv')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,ticket_id,issue_type,issue_description,case_owner,case_owner_description,street_address,city,state,zip_code,neighborhood_district,...,sr_xcoordinate,sr_ycoordinate,latitude,longitude,method_received,sr_priority,goal_days,actual_completed_days,ticket_year,ObjectId
0,23-10000374,BULKY TRASH REQUEST BY APPOINTMENT,NaN,Solid_Waste_Management,Solid Waste Management,21122 SW 92ND CT,Town_of_Cutler_Bay,Florida,33189.0,District 8,...,873554.351,450424.639,25.571308,-80.340525,WEB,STANDARD,8.0,11.0,2023,1
1,23-10000376,BREEDER / PET STORE FOLLOW-UP,NaN,Enforcement_Section-3-36,Enforcement Section,9690 NW 41ST ST,City_of_Doral,Florida,33178.0,District 12,...,869217.668,537584.781,25.811171,-80.352385,INHOUSE,STANDARD,180.0,1.0,2023,2
2,23-10000377,INJURED ANIMAL,NaN,Enforcement_Section-3-36,Enforcement Section,NW 142ND LN & NW 87TH AVE,City_of_Miami_Lakes,Florida,33018.0,District 13,...,872951.375,570986.622,25.903017,-80.340527,INHOUSE,EMERGNCY,180.0,0.0,2023,3
3,23-10000378,BULKY TRASH REQUEST BY APPOINTMENT,NaN,Solid_Waste_Management,Solid Waste Management,10380 SW 66TH TER,Miami_Dade_County,Florida,33173.0,District 10,...,866149.750,499350.780,25.706020,-80.362272,WEB,STANDARD,8.0,28.0,2023,4
4,23-10000379,ILLEGAL DUMPING / LITTER,NaN,COM_SW,COM Solid Waste,1656 SW 6TH ST,City_of_Miami,Florida,33135.0,District 5,...,912160.747,521835.219,25.767201,-80.222142,IPHONE,STANDARD,7.0,16.0,2023,5


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 343851 entries, 0 to 343850
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   ticket_id                      343851 non-null  object 
 1   issue_type                     343851 non-null  object 
 2   issue_description              0 non-null       float64
 3   case_owner                     343851 non-null  object 
 4   case_owner_description         343851 non-null  object 
 5   street_address                 343451 non-null  object 
 6   city                           343379 non-null  object 
 7   state                          343449 non-null  object 
 8   zip_code                       343449 non-null  float64
 9   neighborhood_district          343851 non-null  object 
 10  created_year_month             343851 non-null  int64  
 11  ticket_created_date_time       343851 non-null  object 
 12  ticket__last_update_date_time 

In [11]:
df['issue_type'].value_counts().head(15)

,count
issue_type,
BULKY TRASH REQUEST BY APPOINTMENT,100851
GREEN WASTE CART REQUEST,49908
MOSQUITO INSPECTION REQUEST,18369
RECYCLING BLUE CART ISSUES,15950
BULKY TRASH COMPLAINT,12165
POTHOLE,10984
ILLEGAL DUMPING - WM,10453
PET ACCOUNT UPDATE,9342
GARBAGE COMPLAINT,9058


In [12]:
df['ticket_status'].value_counts()

,count
ticket_status,
CLOSED,311740
CANCEL,16439
LOCKED,8930
DUP,4532
WIP,1314
LIEN,516
OPEN,380


In [13]:
df['created'] = pd.to_datetime(df['ticket_created_date_time'], errors='coerce', utc=True)
df['closed'] = pd.to_datetime(df['ticket_closed_date_time'], errors='coerce', utc=True)
print('Created dates that failed:', df['created'].isna().sum())
print('Closed dates missing:', df['closed'].isna().sum())

df['response_days'] = (df['closed'] - df['created']).dt.total_seconds() / 86400
df['response_days'].describe()

Created dates that failed: 0
Closed dates missing: 10711


,response_days
count,333140.000000
mean,10.817698
std,22.013618
min,-0.852083
25%,1.445139
50%,7.322569
75%,12.101389
max,513.605556


In [14]:
start = len(df)

clean = df.dropna(subset=['created', 'closed', 'zip_code'])
print('Removed for missing dates or ZIP:', start - len(clean))

before = len(clean)
clean = clean[clean['response_days'] >= 0].copy()
print('Removed for closing before opening:', before - len(clean))

before = len(clean)
clean = clean[clean['ticket_status'] == 'CLOSED'].copy()
print('Removed for not being truly closed:', before - len(clean))

clean['zip_code'] = clean['zip_code'].astype(int).astype(str)
print('Rows left:', len(clean))

Removed for missing dates or ZIP: 11081
Removed for closing before opening: 4109
Removed for not being truly closed: 21386
Rows left: 307275


In [15]:
for t in ['POTHOLE', 'ILLEGAL DUMPING - WM', 'ILLEGAL DUMPING / LITTER']:
    s = clean[clean['issue_type'] == t]
    print(t, ':', len(s), 'requests across', s['zip_code'].nunique(), 'ZIP codes')

POTHOLE : 5030 requests across 70 ZIP codes
ILLEGAL DUMPING - WM : 5356 requests across 60 ZIP codes
ILLEGAL DUMPING / LITTER : 3247 requests across 21 ZIP codes


In [16]:
ISSUE = 'POTHOLE'

subset = clean[clean['issue_type'] == ISSUE]
print(ISSUE, ':', len(subset), 'requests')
print('Median days to close:', round(subset['response_days'].median(), 1))

POTHOLE : 5030 requests
Median days to close: 4.3


In [17]:
subset[['response_days', 'actual_completed_days', 'goal_days']].head(10)

,response_days,actual_completed_days,goal_days
150,71.305556,72.0,3.0
193,32.152083,33.0,3.0
226,1.318750,1.0,3.0
274,3.587500,4.0,3.0
276,16.584722,17.0,3.0
280,72.528472,73.0,3.0
291,72.495139,73.0,3.0
299,71.470833,72.0,3.0
310,1.409722,2.0,3.0
317,2.402778,3.0,3.0


In [18]:
df[df['issue_type'] == 'POTHOLE']['ticket_status'].value_counts()

,count
ticket_status,
CLOSED,8139
LOCKED,1525
DUP,990
CANCEL,330
